# 50_vertical_xsection.ipynb — Overview

This notebook constructs vertical cross sections from 3D WRF outputs along a user-defined transect. Using [`wrf-python`](https://wrf-python.readthedocs.io/en/latest/), it interpolates temperature, potential temperature, wind, moisture, divergence/convergence, vorticity, and moisture fluxes along the section and produces annotated plots.

## Structure
- **Setup:** specify domain metadata, file paths, target time, and cross-section endpoints.
- **Load:** gather `wrfout_*` files and open them as `wrfin` for `wrf-python` routines.
- **Derive:** interpolate fields onto the cross section, compute along-section wind, vertical velocity, and derived diagnostics.
- **Plot:** render filled contours with optional wind barbs and terrain shading.

## Usage
1. Update user settings (domain, path, time, start/end coordinates) before running the code.
2. Ensure the requested `wrfout` files exist under `sample_data/Run_WRF/<domain>/<setting>/`.
3. Run the cells sequentially. Add `plt.savefig(...)` where exports are needed.
4. Adjust `num_cross_points`, contour levels, or wind thinning to suit your domain.

## Notes
- Requires `wrf-python`, `netCDF4`, `numpy`, `xarray`, and `matplotlib`.
- Divergence and vorticity are reported in ×10⁵ s⁻¹ for readability.
- Moisture flux is computed as specific humidity times along-section wind; integrate with caution for deep layers.
- Cross-section distance is computed via a great-circle approximation between the endpoints.


Instructions: Configure the user settings below (file paths, target time, cross-section endpoints) before running the subsequent cells.

In [ ]:
from pathlib import Path
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from netCDF4 import Dataset
from wrf import (
    ALL_TIMES,
    CoordPair,
    getvar,
    interplevel,
    interpline,
    latlon_coords,
    to_np,
    vertcross,
    udiv,
    vorticity,
)

# ----------------------
# USER SETTINGS
# ----------------------
domain_name = "Bangkok"
domain_id = "d03"
setting = "test"
data_dir = Path("../../sample_data/Run_WRF") / domain_name / setting
wrf_pattern = f"wrfout_{domain_id}_*"

# Target analysis time (UTC)
target_time = "2025-01-04 06:00:00"

# Cross-section endpoints (lat, lon)
start_point = CoordPair(lat=13.5, lon=100.0)
end_point = CoordPair(lat=14.5, lon=101.0)
num_cross_points = 240  # increase for finer sampling

# Plot styling
wind_skip = (3, 6)  # (vertical, horizontal) thinning for quiver
cmap_default = "coolwarm"


Instructions: The next cell locates `wrfout_*` files, opens them, and determines the closest available time index.

In [ ]:
def find_wrf_files(base_dir: Path, pattern: str):
    files = sorted(base_dir.glob(pattern))
    if not files:
        raise FileNotFoundError(f"No files matched: {base_dir / pattern}")
    return files


def open_wrfin(files):
    return [Dataset(str(path)) for path in files]


def resolve_time_index(wrfin, target_time_str):
    times_da = getvar(wrfin, "times", timeidx=ALL_TIMES)
    times = np.array([np.datetime64(str(t)) for t in to_np(times_da)])

    if target_time_str:
        target = np.datetime64(target_time_str.replace(" ", "T"))
        idx = int(np.argmin(np.abs(times - target)))
    else:
        idx = 0
        target = times[idx]

    return idx, target


wrf_files = find_wrf_files(data_dir, wrf_pattern)
wrfin = open_wrfin(wrf_files)
timeidx, analysis_time = resolve_time_index(wrfin, target_time)
print(f"Loaded {len(wrf_files)} file(s). Using time index {timeidx} at {analysis_time} UTC.")

Instructions: The following helpers compute distances, preload 3D fields, and set up cross-section geometry. Run them prior to extracting diagnostics.

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    r = 6371.0
    dlat = np.deg2rad(lat2 - lat1)
    dlon = np.deg2rad(lon2 - lon1)
    lat1_rad = np.deg2rad(lat1)
    lat2_rad = np.deg2rad(lat2)
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(dlon / 2.0) ** 2
    return 2.0 * r * np.arcsin(np.sqrt(a))


def cumulative_distance(latitudes, longitudes):
    dist = np.zeros_like(latitudes)
    for i in range(1, latitudes.size):
        dist[i] = dist[i - 1] + haversine_km(latitudes[i - 1], longitudes[i - 1], latitudes[i], longitudes[i])
    return dist


def along_section_unit(start_point: CoordPair, end_point: CoordPair):
    dlat = np.deg2rad(end_point.lat - start_point.lat)
    dlon = np.deg2rad(end_point.lon - start_point.lon)
    lat_avg = np.deg2rad((start_point.lat + end_point.lat) / 2.0)
    dx = np.cos(lat_avg) * dlon
    dy = dlat
    mag = np.hypot(dx, dy)
    if mag == 0:
        raise ValueError("Start and end points must differ")
    return dx / mag, dy / mag


def preload_cross_section_fields(wrfin, timeidx):
    fields = {
        "theta": getvar(wrfin, "theta", timeidx=timeidx),
        "temperature": getvar(wrfin, "tk", timeidx=timeidx, units="degC"),
        "qvapor": getvar(wrfin, "QVAPOR", timeidx=timeidx),
        "u": getvar(wrfin, "ua", timeidx=timeidx, units="m s-1"),
        "v": getvar(wrfin, "va", timeidx=timeidx, units="m s-1"),
        "w": getvar(wrfin, "wa", timeidx=timeidx, units="m s-1"),
        "height": getvar(wrfin, "z", timeidx=timeidx, units="m"),
        "terrain": getvar(wrfin, "ter", timeidx=timeidx, units="m"),
        "pressure": getvar(wrfin, "pressure", timeidx=timeidx),
    }
    dx = float(getvar(wrfin, "dx"))
    dy = float(getvar(wrfin, "dy"))
    fields["divergence"] = udiv(fields["u"], fields["v"], dx, dy)
    fields["vorticity"] = vorticity(fields["u"], fields["v"], dx, dy)
    return fields


Instructions: Execute the next cell to derive cross-section fields (temperature, wind, moisture, divergence, vorticity, flux) and associated coordinates.

In [ ]:
def compute_cross_sections(wrfin, fields, start_point, end_point, num_points):
    wrf_ref = wrfin[0]
    theta_x = vertcross(fields["theta"], fields["terrain"], start_point=start_point,
                        end_point=end_point, wrfin=wrf_ref, meta=True, latlon=True)
    temp_x = vertcross(fields["temperature"], fields["terrain"], start_point=start_point,
                       end_point=end_point, wrfin=wrf_ref, meta=True, latlon=True)
    qv_x = vertcross(fields["qvapor"], fields["terrain"], start_point=start_point,
                     end_point=end_point, wrfin=wrf_ref, meta=True, latlon=True)
    u_x = vertcross(fields["u"], fields["terrain"], start_point=start_point,
                    end_point=end_point, wrfin=wrf_ref, meta=True, latlon=True)
    v_x = vertcross(fields["v"], fields["terrain"], start_point=start_point,
                    end_point=end_point, wrfin=wrf_ref, meta=True, latlon=True)
    w_x = vertcross(fields["w"], fields["terrain"], start_point=start_point,
                    end_point=end_point, wrfin=wrf_ref, meta=True, latlon=True)
    div_x = vertcross(fields["divergence"], fields["terrain"], start_point=start_point,
                      end_point=end_point, wrfin=wrf_ref, meta=True, latlon=True) * 1e5
    vort_x = vertcross(fields["vorticity"], fields["terrain"], start_point=start_point,
                       end_point=end_point, wrfin=wrf_ref, meta=True, latlon=True) * 1e5
    height_x = vertcross(fields["height"], fields["terrain"], start_point=start_point,
                         end_point=end_point, wrfin=wrf_ref, meta=True, latlon=True)

    lon_line = np.linspace(start_point.lon, end_point.lon, height_x.sizes[height_x.dims[-1]])
    lat_line = np.linspace(start_point.lat, end_point.lat, height_x.sizes[height_x.dims[-1]])
    distance_km = cumulative_distance(lat_line, lon_line)
    terrain_line = to_np(interpline(fields["terrain"], start_point=start_point,
                                    end_point=end_point, wrfin=wrf_ref, meta=True))

    ux, uy = along_section_unit(start_point, end_point)
    along_wind = u_x * ux + v_x * uy
    moisture_flux = qv_x * along_wind

    cross = {
        "theta": theta_x,
        "temperature": temp_x,
        "specific_humidity": qv_x * 1000.0,
        "specific_humidity_raw": qv_x,
        "divergence": div_x,
        "vorticity": vort_x,
        "along_wind": along_wind,
        "vertical_velocity": w_x,
        "moisture_flux": moisture_flux,
        "height": height_x / 1000.0,  # convert to km for plotting
        "distance_km": distance_km,
        "terrain": terrain_line / 1000.0,
        "lat_line": lat_line,
        "lon_line": lon_line,
    }
    cross["specific_humidity"].attrs["units"] = "g kg^-1"
    cross["divergence"].attrs["units"] = "1e-5 s^-1"
    cross["vorticity"].attrs["units"] = "1e-5 s^-1"
    cross["along_wind"].attrs["units"] = "m s^-1"
    cross["vertical_velocity"].attrs["units"] = "m s^-1"
    cross["moisture_flux"].attrs["units"] = "kg kg^-1 m s^-1"
    cross["height"].attrs["units"] = "km"
    return cross
